# Experimento: Captura Causal (Forward + Backward) en `CompactTrace` (`Qwen/Qwen3.5-0.8B`, Colab L4)

Evalúa el diagnóstico causal con gradientes guardados dentro de `CompactTrace`:
1. Captura `CompactTrace(..., capture_grad=True, grad_mode="compressed", grad_checkpointing=True)` sobre respuestas en bucle de GSM8K (guardando activaciones + gradientes `target` y `healthy` comprimidos).
2. Compara **diagnóstico causal unificado (MLP + Atención)** (`diagnose_causal_traces`) frente a ranking pasivo (`paired_diff_norm`), reportando dónde quedan `L15`/`L19` MLP y `L15` Atención.
3. Mide el **screening 100% offline** (`screen_patches_from_traces`, milisegundos sin GPU) frente al screening causal online (`gradient_screen_candidates`) y el screening forward (`screen_patch_candidates`).

Guarda el resultado de forma atómica en `MyDrive/wsi_runs/FINALES/experiment_causal_diagnose_l4.json` y apaga el runtime solo tras verificar el archivo final.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "experiment_causal_diagnose_l4"
REPO = "/content/wsi"
BASE = "/content/base"
N_SEQS = 8
MAX_NEW_TOK = 1024

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers safetensors pandas pyarrow huggingface_hub hf_transfer
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, json, subprocess, datetime
from google.colab import drive
drive.mount("/content/drive")
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(FINALES, exist_ok=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit:", COMMIT, "| modelo en:", BASE)

In [ ]:
import os, subprocess, sys
LOCAL_OUT = f"/content/{RUN_NAME}.json"
SCRIPT = f"{REPO}/scripts/experiment_causal_diagnose.py"
if not os.path.exists(SCRIPT):
    raise FileNotFoundError(f"Falta {SCRIPT} en {REPO} (commit {COMMIT}). Hay que hacer commit+push a GitHub antes de correr en Colab.")

cmd = [
    sys.executable, "-u", SCRIPT,
    "--model", BASE,
    "--n-seqs", str(N_SEQS),
    "--max-new-tokens", str(MAX_NEW_TOK),
    "--out", LOCAL_OUT,
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

In [ ]:
res = json.load(open(LOCAL_OUT, encoding="utf-8"))
print("=== CAUSAL DIAGNOSIS vs PASSIVE ===")
print("Top causal:", res["diagnosis_summary"])
print("Top passive:", res["passive_top6"])
print("Probe ranks (L15/L19):", res["probe_ranks"])
print("\n=== TRACE SIZE & SPEED ===")
print("Peak VRAM (MB):", res["peak_vram_mb"], "| Capture (s):", res["capture_s"])
print("Mean trace MB:", res["mean_trace_mb"], "| Summary-only grad MB:", res["mean_grad_summary_only_mb"])
print("Timing (s):", res["timing_s"])
print("Correlations:", res["correlations"])

res["_final"] = {
    "complete": True,
    "model": MODEL,
    "commit": COMMIT,
    "notebook": "causal_diagnose_colab.ipynb",
    "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w", encoding="utf-8") as f:
    json.dump(res, f, indent=2)
os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", dst)

In [ ]:
# Apagar runtime solo si el JSON final quedó verificado en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst, encoding="utf-8")).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")